# 🐍 Excel 操作を Python でやってみよう

**今日のゴール**: ふだん Excel でやっている作業を Python で動かして、「**こんなに一瞬で終わるのか!**」を体験すること。

コードを覚える必要はありません。**上から順に実行して、結果を眺める**だけで OK です。

## 🗺️ 今日やること

| | やること | Excel でいうと |
|---|---|---|
| ① | Excel を読む | ファイルを開いて中身を見る |
| ② | 集計する | SUM・フィルター・ピボット・グラフ |
| ③ | 3シートを整えて結合 | 列を入れ替えて、コピペで1つにまとめる |
| ④ | if と for | IF 関数と「同じ作業のくり返し」 |
| ⑤ | **20フォルダ・20ファイルに自動分割** | フィルター → 新しいブック → 保存 を20回 |
| ⑥ | ボーナス: 機械学習 | 訪問件数と広告費から、来月の売上を予測 |

## 💻 Google Colab の使い方

**Colab とは**: ブラウザだけで Python が動く Google のサービス。**インストール不要**で、いま開いているこの画面がそのまま実行環境です。

**最初に1回だけ**
- メニューの **「ファイル → ドライブにコピーを保存」** を押す(自分用のコピーができて、自由に実行・書き換えできます)
- あとは上から実行するだけ。**Excel ファイルのアップロードは不要**です(自動で取ってきます)

**実行のしかた**
- 灰色の箱(**セル**)をクリックして **`Shift + Enter`**(またはセル左の **▶ ボタン**)
- **必ず上から順に**実行する(飛ばすと「そんなもの知りません」というエラーになります)
- 最初の実行だけ、準備に数十秒かかることがあります
- 「このノートブックは Google が作成したものではありません」と出たら **「このまま実行」**

**困ったとき**
- 赤い文字(エラー)が出ても**壊れません**。慌てず、**上のセルから順にもう一度**実行すれば直ります
- どうにもならないときは、メニューの「ランタイム → すべてのセルを実行」

## 📝 コードの読み方(これだけ知っていれば大丈夫)

| 用語 | 意味 | 例 |
|---|---|---|
| **コメント** | `#` から右は人間向けのメモ。Python は読み飛ばす | `# 合計を出す` |
| **変数** | データに付ける名札。`=` は「右のものを左の名前に入れる」 | `total = 100` |
| **関数とカッコ** | 命令のこと。`命令(材料)` の形で書く(Excel の `=SUM(A1:A10)` と同じ) | `print("こんにちは")` |
| **ライブラリ** | 便利な道具箱。`import` で読み込んで使う | `import pandas as pd` |

このノートでは、**すべての行の上に `#` で日本語の説明**を付けています。説明だけ読めば流れが分かります。

## 📊 今日のデータ `sales_data.xlsx`

架空の会社の売上明細です(0章で自動的に用意されます)。

- **3シート**: `2023年度` / `2024年度` / `2025年度`(各年度 4月〜翌3月)
- 各シート **約1000行**、合計 **約3000行**
- **20地区**: 札幌・仙台・新潟・金沢・長野・宇都宮・さいたま・千葉・東京・横浜・静岡・名古屋・京都・大阪・神戸・岡山・広島・高松・福岡・那覇
- **8列**: 売上日 / 地区 / 担当者 / 商品名 / カテゴリ / 単価 / 数量 / 売上金額

⚠️ **2・3シート目は、列の並びがバラバラ**です(担当者が先頭だったり、売上金額が先頭だったり)。現場の Excel でよくあるやつです。あとで Python で整えます。

もう1枚、4枚目に `月次データ` シートがあります(月ごと・地区ごとの**訪問件数**と**広告費**)。最後のボーナスで使います。

## 0. 準備

**この章でやること**: 道具(ライブラリ)を読み込み、今日使う Excel ファイルを用意します。**セルを3つ実行するだけ**です。
Excel でいえば「Excel を起動して、ファイルを開く前」の段階です。

> 💡 **ライブラリ**: Python に機能を足す道具箱。今日の主役は **pandas**(パンダス)= 表を扱う道具で、「Python 版の Excel」のようなものです。

In [ ]:
# グラフで日本語が文字化けしないようにする部品をインストールする(数秒かかります)
%pip install -q matplotlib-fontja
# 表を扱う道具 pandas を pd という短い名前で読み込む(Python 版の Excel のようなもの)
import pandas as pd
# グラフを描く道具を plt という短い名前で読み込む
import matplotlib.pyplot as plt
# グラフの日本語が文字化けしないようにする部品を読み込む
import matplotlib_fontja
# フォルダを作るなど、ファイル操作の道具を読み込む
import os
# インターネットからファイルを取ってくる道具を読み込む
import urllib.request
# フォルダを zip にまとめる道具を読み込む
import shutil
# 時間を計るストップウォッチの道具を読み込む
import time
# 読み込みが終わったことを表示する
print("準備OK!")

### 📥 Excel ファイルを取ってくる

今日使う `sales_data.xlsx` を、インターネットから自動でダウンロードします。**ファイルを選んだりアップロードしたりする必要はありません。**

In [ ]:
# Excel ファイルが置いてある場所(インターネット上の住所)を url という入れ物に入れる
url = "https://raw.githubusercontent.com/Tomoaki-Kotsuka/python-kouza/main/sales_data.xlsx"
# sales_data.xlsx がまだ無いときだけ
if not os.path.exists("sales_data.xlsx"):
    # ダウンロードを試してみる
    try:
        # url の場所から Excel ファイルを取ってきて、sales_data.xlsx という名前で保存する
        urllib.request.urlretrieve(url, "sales_data.xlsx")
        # 取ってこられたことを表示する
        print("sales_data.xlsx をダウンロードしました")
    # うまく取ってこられなかったときは、こちらに来る
    except Exception:
        # 次のセルが同じデータを作るので、メッセージだけ表示する
        print("ダウンロードできませんでした。次のセルが同じデータを作るので、そのまま進んでください")
# すでにあるときは
else:
    # 何もせず、メッセージだけ表示する
    print("sales_data.xlsx はすでにあります")

### 🛟 保険セル(全員そのまま実行)

ダウンロードできなかったときのために、**まったく同じデータをその場で作る**セルです。

- **中身は読まなくて OK**(長いですが、データを作っているだけです)
- ダウンロードできた人は、実行しても何も起きません

In [ ]:
# 乱数(サイコロ)を使うための道具を読み込む
import random
# その月が何日まであるかを調べる道具を読み込む
import calendar
# 表を扱う道具 pandas を pd という名前で読み込む
import pandas as pd


# 売上データの Excel ファイルを作る手順をまとめて、make_sales_data という名前を付ける
def make_sales_data(path="sales_data.xlsx", seed=42):
    # 乱数のタネを固定する(毎回まったく同じデータになる)
    rng = random.Random(seed)
    # 地区ごとの設定: (月の売上の目安[万円], 月の明細件数の目安, 担当者)
    areas = {
        "札幌": (150, 4, ["佐藤 健一", "高橋 美咲"]),
        "仙台": (110, 4, ["伊藤 大輔", "渡辺 由美", "菅原 拓也"]),
        "新潟": (100, 3, ["小林 誠", "五十嵐 恵"]),
        "金沢": (80, 3, ["中村 翔太", "北川 綾香"]),
        "長野": (80, 3, ["宮沢 浩二", "丸山 千尋"]),
        "宇都宮": (80, 3, ["鈴木 直樹", "阿久津 真理"]),
        "さいたま": (160, 4, ["新井 雄太", "関根 沙織", "金子 亮"]),
        "千葉": (140, 4, ["石井 達也", "川島 奈々"]),
        "東京": (420, 8, ["田中 一郎", "山本 さくら", "松本 健太"]),
        "横浜": (240, 6, ["加藤 隼人", "小川 麻衣", "原田 修"]),
        "静岡": (120, 3, ["望月 剛", "杉山 香織"]),
        "名古屋": (260, 6, ["水野 博之", "服部 愛", "近藤 悠"]),
        "京都": (150, 4, ["西村 和也", "奥田 美穂"]),
        "大阪": (330, 7, ["吉田 康平", "森本 絵里", "岡本 大地"]),
        "神戸": (150, 4, ["藤原 智史", "井上 彩"]),
        "岡山": (100, 3, ["三宅 俊介", "難波 裕子"]),
        "広島": (140, 4, ["村上 洋平", "藤井 友香", "山根 徹"]),
        "高松": (80, 3, ["大西 正樹", "真鍋 理恵"]),
        "福岡": (130, 5, ["古賀 竜也", "松尾 美紀", "江口 航"]),
        "那覇": (60, 3, ["比嘉 優", "金城 あかね"]),
    }
    # 商品の一覧: (商品名, カテゴリ, 単価)
    products = [
        ("コピー用紙A4(5000枚)", "文房具", 4200),
        ("ボールペン(10本セット)", "文房具", 1200),
        ("フラットファイル(50冊)", "文房具", 2800),
        ("オフィスチェア", "オフィス家具", 24800),
        ("昇降デスク", "オフィス家具", 49800),
        ("収納キャビネット", "オフィス家具", 32000),
        ("ワイヤレスマウス", "PC周辺機器", 3200),
        ("外付けキーボード", "PC周辺機器", 5800),
        ("27インチモニター", "PC周辺機器", 36000),
        ("Webカメラ", "PC周辺機器", 7400),
        ("シュレッダー", "オフィス機器", 28000),
        ("ラベルプリンター", "オフィス機器", 15800),
    ]
    # 来月(2026年4月)の予定: 地区ごとの (訪問予定件数, 広告費の予算[万円])
    next_plan = {
        "札幌": (42, 25), "仙台": (55, 45), "新潟": (38, 20), "金沢": (58, 43), "長野": (45, 30),
        "宇都宮": (33, 18), "さいたま": (60, 28), "千葉": (46, 52), "東京": (72, 22), "横浜": (40, 35),
        "静岡": (36, 15), "名古屋": (64, 30), "京都": (44, 46), "大阪": (35, 55), "神戸": (30, 24),
        "岡山": (41, 12), "広島": (56, 33), "高松": (28, 26), "福岡": (66, 50), "那覇": (54, 44),
    }
    # 基本の列の並びを決める
    columns = ["売上日", "地区", "担当者", "商品名", "カテゴリ", "単価", "数量", "売上金額"]
    # シートごとの列の並び(2・3シート目はわざとバラバラにする)
    sheet_columns = {
        2023: columns,
        2024: ["担当者", "地区", "売上日", "カテゴリ", "商品名", "数量", "単価", "売上金額"],
        2025: ["売上金額", "商品名", "カテゴリ", "売上日", "数量", "単価", "地区", "担当者"],
    }
    # 年度ごとの明細を入れる箱を用意する
    rows = {2023: [], 2024: [], 2025: []}
    # 月次データ(年月・地区・訪問件数・広告費)を入れる箱を用意する
    cond_rows = []
    # 2023年4月から数えて何か月目かを数える数字(最初は 0)
    t = 0
    # 年度を 2023 → 2024 → 2025 の順にくり返す
    for fy in [2023, 2024, 2025]:
        # 月を 4月 → 翌年3月 の順にくり返す
        for month in [4, 5, 6, 7, 8, 9, 10, 11, 12, 1, 2, 3]:
            # カレンダー上の年を、まず年度と同じにしておく
            year = fy
            # もし 1〜3月なら
            if month <= 3:
                # カレンダー上は翌年になる
                year = fy + 1
            # その月が何日まであるかを調べる
            last_day = calendar.monthrange(year, month)[1]
            # 「2023-04」のような年月の文字を作る
            ym = str(year) + "-" + str(month).zfill(2)
            # 地区を1つずつ取り出してくり返す
            for area in areas:
                # その地区の設定を取り出す
                base, n_base, staff = areas[area]
                # その月に営業担当が顧客を訪問した件数を決める(20〜80件)
                visits = rng.randint(20, 80)
                # その月に使った広告費を決める(10〜60万円)
                ad_cost = rng.randint(10, 60)
                # 条件による売上の倍率は、まず 1.00 倍(ふつうの月)にしておく
                effect = 1.00
                # 訪問件数が多い(50件以上)だけなら
                if visits >= 50 and ad_cost < 40:
                    # 少しだけ増える
                    effect = 1.08
                # 広告費が多い(40万円以上)だけなら
                if visits < 50 and ad_cost >= 40:
                    # 少しだけ増える
                    effect = 1.10
                # 訪問件数も広告費も多い月は
                if visits >= 50 and ad_cost >= 40:
                    # 売上がはっきり跳ねる
                    effect = 1.50
                # 月次データに1行追加する
                cond_rows.append([ym, area, visits, ad_cost])
                # その月の売上の目安 = 規模 × 条件による倍率 × 少しのゆらぎ
                target = base * 10000 * effect * rng.gauss(1, 0.05)
                # その月の明細件数を決める(目安から -1〜+1 件ゆらす)
                n = n_base + rng.choice([-1, 0, 0, 1])
                # 月の売上を明細ごとに分ける割合を入れる箱を用意する
                weights = []
                # その月に売れた商品を入れる箱を用意する
                items = []
                # 明細の件数ぶんくり返す
                for i in range(n):
                    # 割合を少しゆらして箱に追加する
                    weights.append(rng.uniform(0.6, 1.4))
                    # 商品を1つ選んで箱に追加する
                    items.append(rng.choice(products))
                # 商品を単価の高い順に並べる(端数を安い商品で調整するため)
                items.sort(key=lambda item: item[2], reverse=True)
                # 前の明細で合わせきれなかった端数(最初は 0)
                carry = 0
                # 明細を1件ずつ作る
                for i in range(n):
                    # 商品名・カテゴリ・単価を取り出す
                    name, category, price = items[i]
                    # この明細で売りたい金額(前の端数も足す)
                    amount = target * weights[i] / sum(weights) + carry
                    # 目安の金額になるように数量を決める(最低1個)
                    qty = max(1, round(amount / price))
                    # 合わせきれなかった端数を次の明細に回す
                    carry = amount - price * qty
                    # 売上日を決める
                    day = pd.Timestamp(year, month, rng.randint(1, last_day))
                    # 担当者を選ぶ
                    person = rng.choice(staff)
                    # 1行分のデータを箱に追加する(売上金額 = 単価 × 数量)
                    rows[fy].append([day, area, person, name, category, price, qty, price * qty])
            # 1か月進める
            t = t + 1
    # 来月(2026年4月)の予定を、地区ごとに月次データへ追加する
    for area in areas:
        # その地区の訪問予定件数と広告費の予算を取り出す
        visits, ad_cost = next_plan[area]
        # 月次データに1行追加する(売上はまだ無い)
        cond_rows.append(["2026-04", area, visits, ad_cost])
    # Excel ファイルを書き込み用に開く
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # 年度ごとに1シートずつ書き込む
        for fy in [2023, 2024, 2025]:
            # 箱の中身を表にする
            df = pd.DataFrame(rows[fy], columns=columns)
            # 売上日の古い順に並べ替える
            df = df.sort_values("売上日", kind="stable")
            # 列をそのシート用の並びにする
            df = df[sheet_columns[fy]]
            # シート名を「2023年度」のように決める
            sheet_name = str(fy) + "年度"
            # シートに書き込む
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            # A〜H 列を1つずつ取り出してくり返す
            for letter in "ABCDEFGH":
                # 列の幅を広げて見やすくする
                writer.sheets[sheet_name].column_dimensions[letter].width = 16
            # 売上日の列が左から何番目(A〜H)かを調べる
            date_letter = "ABCDEFGH"[sheet_columns[fy].index("売上日")]
            # 売上日の列のセルを、見出しをのぞいて1つずつ取り出す
            for cell in writer.sheets[sheet_name][date_letter][1:]:
                # 日付を 2023/04/01 の形で表示する設定にする
                cell.number_format = "yyyy/mm/dd"
            # 商品名の列が左から何番目(A〜H)かを調べる
            name_letter = "ABCDEFGH"[sheet_columns[fy].index("商品名")]
            # 商品名の列だけさらに広げる
            writer.sheets[sheet_name].column_dimensions[name_letter].width = 26
        # 月次データの箱の中身を表にする
        df_cond = pd.DataFrame(cond_rows, columns=["年月", "地区", "訪問件数", "広告費(万円)"])
        # 4枚目のシート「月次データ」に書き込む
        df_cond.to_excel(writer, sheet_name="月次データ", index=False)
        # A〜D 列を1つずつ取り出してくり返す
        for letter in "ABCD":
            # 列の幅を広げて見やすくする
            writer.sheets["月次データ"].column_dimensions[letter].width = 14

# sales_data.xlsx がまだ無いときだけ
if not os.path.exists("sales_data.xlsx"):
    # 上で用意した手順を使って、同じデータの Excel ファイルを作る
    make_sales_data("sales_data.xlsx")
    # 作ったことを表示する
    print("sales_data.xlsx をその場で作りました")
# すでにあるときは
else:
    # 何もせず、メッセージだけ表示する
    print("sales_data.xlsx は用意できています")

## 1. pandas で Excel を読む

**この章でやること**: 1シート目(2023年度)を読み込んで、中身をざっと確認します。

**Excel だと**: 開いて、下までスクロールして、行数を数えて、列の中身を目で確認…

**ここで覚える命令**
- `pd.read_excel()` … Excel を読み込む
- `.head()` … 先頭の5行を見る
- `.shape` … 行数と列数を見る

> 💡 **DataFrame(データフレーム)**: pandas が扱う「表」のこと。**Excel のシート1枚**だと思ってください。
> このノートでは表に `df` という名前(変数)を付けています。
>
> `df = pd.read_excel(...)` は「Excel を読み込んで、その表に `df` という名札を付ける」という意味です。

In [ ]:
# 読み込む Excel ファイルの名前を file という入れ物に入れておく
file = "sales_data.xlsx"
# Excel の「2023年度」シートを読み込んで、df という名前の表にする
df = pd.read_excel(file, sheet_name="2023年度")
# 表の先頭5行を表示して、読み込めたか確認する
display(df.head())

In [ ]:
# 表の大きさ(行数, 列数)を表示する。Excel で最終行までスクロールして数える代わり
print(df.shape)

In [ ]:
# 列の名前と、各列に入っているデータの種類(日付・文字・数字)を一覧で表示する
df.info()

In [ ]:
# 数字の列の平均・最小・最大などをまとめて表示する(AVERAGE や MAX を全列に一気にかけるイメージ)
display(df.describe())

## 2. 基本の集計

**この章でやること**: 合計・平均、地区別の集計、並べ替え、絞り込み、ピボット、グラフを一気にやります。

**Excel だと**: SUM 関数、SUMIF、並べ替え、フィルター、ピボットテーブルの画面操作…と、それぞれ別の操作が必要。

**ここで覚える命令**
- `df["列名"]` … 列を1つ取り出す
- `.groupby("地区")` … 地区ごとにまとめる(SUMIF を全地区ぶん一気に)
- `df[条件]` … 条件に合う行だけ取り出す(フィルター)

In [ ]:
# 「売上金額」列の合計を出す(Excel の SUM)
total = df["売上金額"].sum()
# 「売上金額」列の平均を出す(Excel の AVERAGE)
mean = df["売上金額"].mean()
# 「売上金額」列の最大を出す(Excel の MAX)
biggest = df["売上金額"].max()
# 合計を、3桁区切りのカンマ付きで表示する
print("合計:", f"{total:,}", "円")
# 平均を、小数を切り捨ててカンマ付きで表示する
print("平均:", f"{mean:,.0f}", "円")
# 最大を、カンマ付きで表示する
print("最大:", f"{biggest:,}", "円")

### 地区別に集計して、並べ替える

In [ ]:
# 地区ごとにグループ分けして、売上金額を合計する(Excel の SUMIF を20地区ぶん一気に)
area_sales = df.groupby("地区")["売上金額"].sum()
# 地区別の合計を表示して確認する
display(area_sales)

In [ ]:
# 売上の大きい順に並べ替える(Excel の「降順で並べ替え」)
area_sales = area_sales.sort_values(ascending=False)
# 上位5地区を表示して確認する
display(area_sales.head(5))

### 条件で絞り込む(フィルター)

> 💡 `==` は「等しい」、`>=` は「以上」。`=` 1つは「名前を付ける」なので、**比べるときは `==` と2つ**書きます。

In [ ]:
# 地区が「東京」の行だけを取り出す(Excel のフィルター)
df_tokyo = df[df["地区"] == "東京"]
# 取り出した表の先頭5行を表示して確認する
display(df_tokyo.head())
# 何行あったかを表示して確認する
print(df_tokyo.shape)

In [ ]:
# 売上金額が 50万円以上の行だけを取り出す(Excel の数値フィルター「以上」)
df_big = df[df["売上金額"] >= 500000]
# 取り出した表の先頭5行を表示して確認する
display(df_big.head())
# 何行あったかを表示して確認する
print(df_big.shape)

### ピボットテーブルとグラフ

In [ ]:
# 行に地区、列にカテゴリを置いて、売上金額を合計する(Excel のピボットテーブル)
pivot = pd.pivot_table(df, index="地区", columns="カテゴリ", values="売上金額", aggfunc="sum")
# できたピボット表を表示して確認する
display(pivot)

In [ ]:
# 金額を 10000 で割って、万円単位にした表を作る
area_sales_man = area_sales / 10000
# 万円単位の上位5地区を表示して確認する
display(area_sales_man.head(5))
# 地区別の売上を棒グラフにする(Excel の「挿入 → グラフ」)
area_sales_man.plot(kind="bar", figsize=(10, 4), title="2023年度 地区別売上")
# たて軸に名前を付ける
plt.ylabel("売上金額(万円)")
# グラフを表示する
plt.show()

## 3. 3つのシートを1つにまとめる

**この章でやること**: 2・3シート目を読み、バラバラな列の並びを揃えてから、3年分を縦に1つの表にまとめます。

**Excel だと**: 列を1つずつ切り取って入れ替え、コピーして下に貼り付け… 1列ずれたら全部やり直し。

**ここで覚える命令**
- `df[columns]` … 列を指定した順番に並べ直す
- `pd.concat()` … 表を縦につなげる

In [ ]:
# 「2024年度」シートを読み込んで、df_2024 という表にする
df_2024 = pd.read_excel(file, sheet_name="2024年度")
# 先頭5行を表示して確認する(列の並びが1シート目と違うことに注目)
display(df_2024.head())

In [ ]:
# 「2025年度」シートを読み込んで、df_2025 という表にする
df_2025 = pd.read_excel(file, sheet_name="2025年度")
# 先頭5行を表示して確認する(こちらも並びが違う)
display(df_2025.head())

### 列の並びを比べて、揃える

3つのシートの列名を並べて表示します。**同じ列がそろっているのに、順番だけが違う**ことを確認しましょう。

In [ ]:
# 2023年度の列名を左から順に表示する
print(list(df.columns))
# 2024年度の列名を左から順に表示する
print(list(df_2024.columns))
# 2025年度の列名を左から順に表示する
print(list(df_2025.columns))

In [ ]:
# お手本にする列の並び(2023年度の並び)を columns という入れ物に入れる
columns = list(df.columns)
# 2024年度の表を、お手本と同じ列の並びに入れ替える
df_2024 = df_2024[columns]
# 2025年度の表を、お手本と同じ列の並びに入れ替える
df_2025 = df_2025[columns]
# 2024年度の先頭5行を表示して、並びが揃ったか確認する
display(df_2024.head())
# 2025年度の先頭5行を表示して、並びが揃ったか確認する
display(df_2025.head())

### 「年度」列を足して、縦につなげる

1つにまとめると、どの行が何年度か分からなくなるので、先に「年度」列を足しておきます。

In [ ]:
# 2023年度の表に「年度」列を追加して、全部の行に 2023 を入れる
df["年度"] = 2023
# 2024年度の表に「年度」列を追加して、全部の行に 2024 を入れる
df_2024["年度"] = 2024
# 2025年度の表に「年度」列を追加して、全部の行に 2025 を入れる
df_2025["年度"] = 2025
# 右端に「年度」列が増えたか、先頭5行を表示して確認する
display(df.head())

In [ ]:
# 3つの表を縦につなげて、df_all という1つの表にする(コピーして下に貼り付ける作業)
df_all = pd.concat([df, df_2024, df_2025], ignore_index=True)
# 先頭5行を表示して確認する
display(df_all.head())
# 末尾5行を表示して、2025年度のデータまで入っているか確認する
display(df_all.tail())
# 全部で何行・何列になったかを表示して確認する
print(df_all.shape)

In [ ]:
# 年度ごとの行数を数えて、3年分がそろっているか確認する
display(df_all["年度"].value_counts())

## 4. if と for を軽く

**この章でやること**: プログラミングの2大基本、**くり返し(for)** と **もし〜なら(if)** を体験します。

**Excel だと**: 同じ操作を20回手でくり返す / IF 関数を書く。Python では「20回やって」と頼めます。

### 🔁 for … くり返し

```
for 名前 in リスト:      ← 最後に「:」(コロン)
    くり返したい処理     ← 行の頭を4文字ぶん下げる(インデント)
```

- リストの中身を**1つずつ**取り出して、下げた行を**くり返し**ます
- **字下げされている行だけ**がくり返しの対象。字下げが「ここからここまで」の目印です

In [ ]:
# for: 「東京」「大阪」「福岡」を順番に1つずつ area に入れて、下の行をくり返す
for area in ["東京", "大阪", "福岡"]:
    # いま area に入っている地区名に「支店」を付けて表示する
    print(area + "支店")

### 🔀 if … もし〜なら

```
if 条件:                ← 最後に「:」(コロン)
    条件に合うときの処理   ← 字下げ
else:                   ← 「そうでなければ」。ここも「:」
    合わないときの処理     ← 字下げ
```

Excel の `=IF(条件, 合うとき, 合わないとき)` と同じ考え方です。

In [ ]:
# ためしに、売上を 120万円 としておく
sales = 1200000
# if: もし売上が 100万円以上なら
if sales >= 1000000:
    # 「◎好調」と表示する
    print("◎好調")
# そうでなければ
else:
    # 「△要フォロー」と表示する
    print("△要フォロー")

### 🔁+🔀 for と if を組み合わせる

20地区すべてについて「3年間の売上が全地区の平均以上なら **◎好調**、そうでなければ **△要フォロー**」と判定します。

```
for 地区 in 20地区:          ← 20回くり返す
    if 売上 >= 基準:         ← そのたびに判定(for の中なので字下げ)
        「◎好調」            ← if の中なので、さらに字下げ
    else:
        「△要フォロー」
```

In [ ]:
# 3年分の表から、地区ごとの売上合計を出す
area_total = df_all.groupby("地区")["売上金額"].sum()
# 地区別の合計を表示して確認する
display(area_total)
# 判定の基準にする金額(全地区の平均)を出す
border = area_total.mean()
# 基準の金額を表示して確認する
print("基準:", f"{border:,.0f}", "円")

In [ ]:
# for: 地区名を1つずつ area に入れて、下の行を20回くり返す
for area in area_total.index:
    # その地区の売上合計を取り出す
    sales = area_total[area]
    # if: もし売上が基準以上なら
    if sales >= border:
        # 地区名と金額と「◎好調」を表示する
        print(area, f"{sales:,}", "円 ◎好調")
    # そうでなければ
    else:
        # 地区名と金額と「△要フォロー」を表示する
        print(area, f"{sales:,}", "円 △要フォロー")

## 5. 仕上げ: 地区ごとにフォルダとファイルを自動で作る

**この章でやること**: 3年分の表を地区ごとに分け、地区フォルダを作って、その中に地区別の Excel を保存します(**20フォルダ・20ファイル**)。

**Excel だと**: 「フィルター → コピー → 新しいブックに貼り付け → フォルダを作る → 名前を付けて保存」を **20回**。

**ここで覚える命令**
- `os.makedirs()` … フォルダを作る
- `.to_excel()` … 表を Excel ファイルとして保存する

🤔 **実行する前に予想してみましょう。手作業なら何分? Python なら何秒?**

In [ ]:
# 「地区」列から、重複をのぞいた地区名の一覧を取り出す(Excel の「重複の削除」)
areas = df_all["地区"].unique()
# 地区名の一覧を表示して確認する
print(areas)
# 地区がいくつあるかを表示して確認する
print(len(areas), "地区")

### ⏱️ いよいよ実行!

ここまでに出てきた **for(くり返し)** と **絞り込み** を組み合わせるだけです。ストップウォッチで時間も計ります。

In [ ]:
# ストップウォッチをスタートする(いまの時刻を覚えておく)
start = time.time()
# 保存先の大もとのフォルダ名を決める
out_dir = "地区別"
# for: 地区名を1つずつ area に入れて、下の行を20回くり返す
for area in areas:
    # その地区の行だけを取り出す(フィルター)
    df_area = df_all[df_all["地区"] == area]
    # 「地区別/東京」のようなフォルダの場所を組み立てる
    folder = os.path.join(out_dir, area)
    # そのフォルダを作る(すでにあってもエラーにしない)
    os.makedirs(folder, exist_ok=True)
    # 「地区別/東京/東京_売上.xlsx」のようなファイルの場所を組み立てる
    path = os.path.join(folder, area + "_売上.xlsx")
    # 取り出した表を Excel ファイルとして保存する(名前を付けて保存)
    df_area.to_excel(path, index=False)
    # 保存した地区名と行数を表示して確認する
    print(area, df_area.shape, "を保存しました")
# ストップウォッチを止めて、かかった秒数を計算する
elapsed = time.time() - start
# かかった時間を表示する
print(f"{len(areas)}地区ぶんのフォルダとファイルを {elapsed:.1f} 秒で完了!")

### できあがりを確認する

画面左の 📁(フォルダのアイコン)を押すと、`地区別` フォルダができているのが見えます。

In [ ]:
# 「地区別」フォルダの中にあるフォルダ名を、名前順に並べて取り出す
folders = sorted(os.listdir(out_dir))
# for: フォルダ名を1つずつ取り出してくり返す
for folder in folders:
    # そのフォルダの中にあるファイル名を調べる
    file_names = os.listdir(os.path.join(out_dir, folder))
    # フォルダ名と、中のファイル名を表示する
    print(folder, "→", file_names)
# フォルダがいくつできたかを表示して確認する
print(len(folders), "フォルダできました")

In [ ]:
# 保存したファイルを1つ読み直して、中身が正しいか確認する(福岡の例)
df_check = pd.read_excel("地区別/福岡/福岡_売上.xlsx")
# 先頭5行を表示して確認する
display(df_check.head())
# 行数と列数を表示して確認する
print(df_check.shape)

### 📦 zip にまとめてダウンロード

ダウンロードした zip を自分の PC で開いて、地区ごとの Excel ができていることを確かめてみましょう。

In [ ]:
# 「地区別」フォルダを丸ごと「地区別.zip」という1つのファイルにまとめる
shutil.make_archive("地区別", "zip", out_dir)
# zip ファイルができたことを表示する
print("地区別.zip を作りました")
# Colab のときだけ、自分の PC にダウンロードする
try:
    # Colab 専用の、ファイルをやり取りする道具を読み込む
    from google.colab import files
    # zip ファイルを自分の PC にダウンロードする
    files.download("地区別.zip")
# Colab 以外で動かしているときは、こちらに来る
except ImportError:
    # ダウンロードは不要なので、メッセージだけ表示する
    print("Colab ではないのでダウンロードは飛ばします(同じフォルダに 地区別.zip があります)")

## ⏱️ 振り返り: 手作業だと? Python だと?

| | やること | かかる時間 |
|---|---|---|
| ✋ **手作業** | フィルター → コピー → 新しいブック → フォルダ作成 → 保存 を **20回** | 1地区1分でも **約20分** |
| 🐍 **Python** | セルを1つ実行 | **さっき表示された秒数** |

しかも Python なら、地区が 20 から **200 に増えてもコードは1文字も変わりません**。来月また同じ作業が来ても、実行し直すだけです。

## 🆚 同じ処理を VBA でも用意しました

Excel には昔から **VBA(マクロ)** という自動化の仕組みがあります。3章 + 5章と同じ処理(3シートの列を揃えて結合 → 20フォルダ・20ファイルに分割)を VBA でも書いてみました。**講師が実行してみます。**

**見てほしいポイント**
- 📏 **コードの量**: 同じことをするのに何行くらい必要か
- ⏱️ **かかる時間**: 実行して計ってみます

VBA は **Excel だけで完結する**のが強みで、今も現場でたくさん使われています。どちらが上という話ではなく、**同じ仕事でも道具によって書く量と速さが変わる**、というところを見てください。

## 6. 【ボーナス】機械学習: 来月の売上を「条件」から予測する

ここは**眺めるだけで OK** です。

1. **売上以外の条件**を2つ使います: その月の **訪問件数**(営業担当が顧客を訪問した件数)と **広告費**(その地区が使った広告費、万円)
2. 過去3年ぶん(36か月 × 20地区)を AI に見せて、**「どんな条件のときに売上が上がるか」を AI 自身に見つけさせます**
3. 見つけたルールに来月(2026年4月)の訪問予定と広告予算を当てはめて、地区ごとの売上を予測します

🤔 **訪問と広告、売上に効くのはどっち? それとも…?**

In [ ]:
# 売上以外の条件(訪問件数・広告費)を、4枚目のシートから読む。最後の20行は来月の予定
cond = pd.read_excel(file, sheet_name="月次データ")

# 3年分の明細を「年月 × 地区」の月次売上に集計して、条件とくっつける
ym = df_all["売上日"].dt.strftime("%Y-%m").rename("年月")
monthly = df_all.groupby([ym, "地区"])["売上金額"].sum().reset_index()
data = cond.merge(monthly, on=["年月", "地区"], how="left")

# 地区の規模の差をならすため「その地区のふだんの月(36か月の中央値)の何倍売れたか」に直す
data["ふだんの売上"] = data.groupby("地区")["売上金額"].transform("median")
data["ふだん比"] = data["売上金額"] / data["ふだんの売上"]
past = data[data["売上金額"].notna()]
future = data[data["売上金額"].isna()].copy()
display(past.head())
print("学習に使う過去データ:", past.shape, " 予測したい来月:", future.shape)

### 🌳 AI に「売上が上がる条件」を見つけさせる

**決定木**という方法を使います。AI がデータを見て、4章でやった **if の条件を自分で組み立てます**。

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

# 条件(訪問件数・広告費)から「ふだんの月の何倍売れるか」を学習する
features = ["訪問件数", "広告費(万円)"]
model = DecisionTreeRegressor(max_depth=2, random_state=0).fit(past[features], past["ふだん比"])

# AI が見つけたルールを図にする
plt.figure(figsize=(12, 5))
plot_tree(model, feature_names=features, filled=True, impurity=False, precision=2, fontsize=12)
plt.title("AI が見つけたルール(value = その地区のふだんの月の何倍売れるか)")
plt.show()

**図の読み方**
- 箱の1行目が **if の条件**。当てはまれば **左(True)**、当てはまらなければ **右(False)** へ進む
- `<= 39.5` のような半端な数字は「39 と 40 の間で分ける」という意味。つまり **右へ進むのは 40 以上**のとき
- いちばん下の箱の `value` が答え = **ふだんの月の何倍売れるか**。色が濃いほど大きい

👀 **いちばん色が濃い箱は、どんな条件の組み合わせでしょう?**

In [ ]:
# 来月(2026年4月)の訪問予定と広告予算を、AI が見つけたルールに当てはめて予測する
future["予測倍率"] = model.predict(future[features])
future["来月の予測"] = future["予測倍率"] * future["ふだんの売上"]
future["伸び率(%)"] = (future["予測倍率"] - 1) * 100
result = future[["地区", "訪問件数", "広告費(万円)", "ふだんの売上", "来月の予測", "伸び率(%)"]]
result = result.sort_values(["伸び率(%)", "来月の予測"], ascending=False).reset_index(drop=True)
display(result.style.format({"ふだんの売上": "{:,.0f}", "来月の予測": "{:,.0f}", "伸び率(%)": "{:+.1f}"}))

top = result[result["伸び率(%)"] == result["伸び率(%)"].max()]
print("売上規模が最大の地区:", result.sort_values("ふだんの売上", ascending=False).iloc[0]["地区"])
print("来月いちばん伸びる地区:", "・".join(top["地区"]), f"(ふだんの月より {top.iloc[0]['伸び率(%)']:+.1f}%)")

# ふだんの月の売上と、来月の予測を並べたグラフ
chart = result.set_index("地区")[["ふだんの売上", "来月の予測"]] / 10000
chart.plot(kind="bar", figsize=(12, 4.5), title="ふだんの月の売上 と 来月(2026年4月)の予測")
plt.ylabel("売上金額(万円)")
plt.show()

## 🎉 まとめ

**今日できるようになったこと**
- ✅ Excel ファイルを Python で読んで、中身を確認する
- ✅ 合計・地区別集計・並べ替え・絞り込み・ピボット・グラフ
- ✅ 列の並びがバラバラなシートを揃えて、1つの表にまとめる
- ✅ for と if で「くり返し」と「条件分け」
- ✅ 20フォルダ・20ファイルを数秒で自動作成
- ✅ (ボーナス)機械学習で、条件の組み合わせから来月の売上を予測

**Python が得意な仕事**: 「同じ作業を何度もくり返す」「ファイルやシートがたくさんある」「毎月同じことをする」

## 🚀 次の一歩

- **自分の Excel で試す**: このノートのファイル名と列名を書き換えれば、自分のデータでも動きます(社外に出せないデータは Colab に上げないよう注意)
- **AI に聞く**: 「この Excel の ◯◯ 列を地区ごとに集計して、別々のファイルに保存する Python コードを書いて」のように、**やりたいことを日本語で伝える**だけでコードを書いてくれます
- **今日のノートを見返す**: 「ドライブにコピー」したノートは自分の Google ドライブに残っています

コードを暗記する必要はありません。**「Python ならこれができる」と知っていること**が一番の武器です。

### お疲れさまでした! 👏